# 📆 Monthly Summary – Flat & Furious

This notebook generates a monthly summary of the group's cycling activity, with rankings, highlights, group progress, and some fun facts to motivate everyone in the peloton.

---


### ✅ What this summary includes

- Total distance of the group this month and year
- Top 3 longest individual rides
- Monthly and annual distance rankings
- Max speed of the month
- % of a full loop around the Earth
- Most common distances (word cloud)
- 2 curiosities (fun equivalents)
- Who disappeared or barely showed up

---


### 1. 📦 Load Required Libraries

In [1]:
import pandas as pd
from datetime import datetime
import matplotlib.pyplot as plt
from wordcloud import WordCloud
from curiosidades import gerar_curiosidades
import os
from dotenv import load_dotenv
from tabulate import tabulate

### 2. ⚙️ Load Environment Variables

In [2]:
load_dotenv()
FORMATTED_PATH = os.getenv("FORMATTED_PATH")
OUTPUT_FOLDER = os.getenv("OUTPUT_FOLDER")

### 3. 🔁 Load activity data

In [3]:
df = pd.read_csv(FORMATTED_PATH)

# 📅 Prepare time-related helper columns
df['date'] = pd.to_datetime(df['date'])  # Ensure datetime format

# Create only the necessary time features
df['month_year'] = df['date'].dt.to_period('M').astype(str)  # e.g., "2025-06"
df['weekday'] = df['date'].dt.dayofweek  # Monday = 0, Sunday = 6

In [4]:
# 📥 Ask the user for the month to analyze
user_input = input("🗓️ Enter the month to analyze (format: YYYY-MM): ").strip()

# Validate and apply filter
try:
    selected_month_year = pd.to_datetime(user_input, format="%Y-%m").strftime('%Y-%m')
    df_selected_month = df[df['month_year'] == selected_month_year]
    print(f"✅ Data filtered for: {selected_month_year}")
except ValueError:
    print("❌ Invalid format. Please use YYYY-MM (e.g., 2025-06).")

🗓️ Enter the month to analyze (format: YYYY-MM): 2025-08
✅ Data filtered for: 2025-08


### 4.⚡ Max Speed of the Month

In [5]:
# 📊 Filter data for the selected month
df_month = df[df['month_year'] == selected_month_year]

# ⚡ Find the athlete with the highest max speed in that month
if not df_month.empty:
    max_speed_month = df_month.loc[df_month['max_speed'].idxmax()]
    print(f"⚡ {max_speed_month['athlete']} was the fastest, hitting {max_speed_month['max_speed']:.1f} km/h.")
else:
    print("⚡ No rides recorded this month.")

⚡ Vinicius Cainelli was the fastest, hitting 54.8 km/h.


In [6]:
# 📊 Filtra dados para o mês selecionado
df_month = df[df['month_year'] == selected_month_year]

# 🏆 Encontra o atleta com mais tempo de atividade (soma de moving_time)
if not df_month.empty:
    # Converter caso moving_time esteja como string (hh:mm:ss)
    df_month['moving_time'] = pd.to_timedelta(df_month['moving_time'])
    # Agrupa por atleta somando o tempo
    total_time = df_month.groupby('athlete', as_index=False)['moving_time'].sum()
    # Busca o atleta com maior tempo total
    top_athlete = total_time.loc[total_time['moving_time'].idxmax()]
    # Extrai horas e minutos
    td = top_athlete['moving_time']
    total_seconds = int(td.total_seconds())
    horas = total_seconds // 3600
    minutos = (total_seconds % 3600) // 60
    print(f"🏆 {top_athlete['athlete']} foi quem mais pedalou, somando {horas}h{minutos:02d}min de tempo de atividade no mês.")
else:
    print("🏆 Nenhuma atividade registrada neste mês.")


🏆 Diego Galdino foi quem mais pedalou, somando 17h18min de tempo de atividade no mês.


C:\Users\dsgal\AppData\Local\Temp\ipykernel_20492\2166092808.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_month['moving_time'] = pd.to_timedelta(df_month['moving_time'])


### 5. 🌍 Distance Around the Earth

In [7]:
# 📅 Get current year

today = datetime.today()
current_year = today.year 

# 📊 Filter rides that occurred in the current year
df_year = df[df['date'].dt.year == current_year]

# ➕ Sum total distance for the year
dist_total_year = df_year['distance'].sum()

# 🌍 Define Earth's circumference in kilometers
around_earth_km = 40075

# ➗ Calculate the percentage of an Earth circumnavigation
percentual = dist_total_year / around_earth_km

# 🖨️ Display result
print(f"🌍 The group has completed {percentual:.2%} of a trip around the Earth "
      f"({dist_total_year:.0f} km ridden in {current_year}).")

🌍 The group has completed 13.31% of a trip around the Earth (5334 km ridden in 2025).


### 6. 🎉 Fun Facts of the Month

In [8]:
distance_month = df_month['distance'].sum()
curiosity = gerar_curiosidades(distance_month, selected_month_year)

print(f"🎉 Fun facts for {selected_month_year}:")
for phrase in curiosity:
    print("•", phrase)


🎉 Fun facts for 2025-08:
• ⛰️ Isso daria 188.5 montes Everest empilhando garrafas de Grolsch.
• 🐜 Daria pra fazer uma fila com 166,818,000 formigas marchando sem parar.


### 7. 📅 Total Distance This Month & Year

In [9]:
print(f"📅 Total group distance this month: {distance_month:.0f} km")
print(f"📆 Total group distance this year: {dist_total_year:.0f} km")


📅 Total group distance this month: 1668 km
📆 Total group distance this year: 5334 km


### 8. 🏁 Top 3 Longest Individual Rides

In [10]:
# 🥇 Get the top 3 longest individual rides for the selected month
top3 = df_month.sort_values(by="distance", ascending=False).head(3)

# 🖨️ Display the results
print("🏁 Top 3 longest individual rides this month:")
for idx, row in enumerate(top3.itertuples(), start=1):
    print(f"{idx}. {row.athlete} – {row.distance:.1f} km")


🏁 Top 3 longest individual rides this month:
1. Diego Galdino – 151.7 km
2. Vinicius Cainelli – 150.9 km
3. Thamiris Ramos – 150.8 km


### 9. 🏆 Rankings – Monthly & Yearly

In [11]:
ranking_month = df_month.groupby('athlete')['distance'].sum().sort_values(ascending=False).reset_index()
ranking_year = df_year.groupby('athlete')['distance'].sum().sort_values(ascending=False).reset_index()

# MONTHLY RANKING
ranking_month = ranking_month.groupby('athlete')['distance'].sum().sort_values(ascending=False).reset_index()
ranking_month.insert(0, '🏅', range(1, len(ranking_month)+1))  # insere coluna 1, 2, 3...
print("📌 Monthly ranking:")
print(tabulate(ranking_month, headers='keys', tablefmt='github', showindex=False))

# ANNUAL RANKING
ranking_year = df_year.groupby('athlete')['distance'].sum().sort_values(ascending=False).reset_index()
ranking_year.insert(0, '🏅', range(1, len(ranking_year)+1))
print("\n🏆 Annual ranking:")
print(tabulate(ranking_year, headers='keys', tablefmt='github', showindex=False))


📌 Monthly ranking:
|   🏅 | athlete              |   distance |
|------|----------------------|------------|
|    1 | Diego Galdino        |     353.19 |
|    2 | Rodrigo Bonilha      |     347.42 |
|    3 | Jair de Souza Junior |     329.14 |
|    4 | Thamiris Ramos       |     322.38 |
|    5 | Vinicius Cainelli    |     302.91 |
|    6 | Jonathan Machado     |      13.14 |

🏆 Annual ranking:
|   🏅 | athlete              |   distance |
|------|----------------------|------------|
|    1 | Rodrigo Bonilha      |    1545.35 |
|    2 | Diego Galdino        |    1100.66 |
|    3 | Thamiris Ramos       |     896.98 |
|    4 | Jair de Souza Junior |     843.54 |
|    5 | Vinicius Cainelli    |     762.6  |
|    6 | Jonathan Machado     |     184.99 |


### 10. 🧍‍♂️ Participation & Engagement Summary

In [12]:
# 📊 Count number of activities and total distance per athlete
activity_counts = df_month['athlete'].value_counts()
distance_sum = df_month.groupby('athlete')['distance'].sum()

# 🧮 Create summary DataFrame
df_stats = pd.DataFrame({'activities': activity_counts, 'total_km': distance_sum}).fillna(0)

# 🔍 Find the athlete with the fewest activities (and lowest distance as tiebreaker)
least = df_stats.sort_values(by=['activities', 'total_km']).head(1)

# 🖨️ Display result
print("😴 Rider with the fewest activities this month:")
for name, row in least.iterrows():
    print(f"• {name} – {int(row['activities'])} rides, {row['total_km']:.1f} km")

😴 Rider with the fewest activities this month:
• Jonathan Machado – 1 rides, 13.1 km


### 11. 🧑‍🤝‍🧑 Menor presença nos pedais de grupo (fins de semana)

In [13]:
# 🎯 Filter rides that happened on Saturday (5) or Sunday (6)
df_weekend = df_month[df_month['date'].dt.dayofweek >= 5]

# 📊 Count how many weekend rides each athlete did
weekend_counts = df_weekend['athlete'].value_counts()

# 📋 Get the full list of athletes who rode this month
athletes_full_list = sorted(df_month['athlete'].unique().tolist())

# 🧮 Create DataFrame with weekend ride counts (0 for those who didn't join)
df_weekend_stats = pd.DataFrame({
    'weekend_rides': weekend_counts
}).reindex(athletes_full_list).fillna(0)

# 🔍 Find the athlete with the fewest weekend rides
least_weekend = df_weekend_stats.sort_values(by='weekend_rides').head(1)

# 🖨️ Display the result
print("\n📉 Least active on group rides (weekends):")
for name, row in least_weekend.iterrows():
    print(f"• {name} – {int(row['weekend_rides'])} weekend rides")


📉 Least active on group rides (weekends):
• Jonathan Machado – 0 weekend rides


In [14]:
from tabulate import tabulate

# 🔷 Título principal
print("\n" + "="*60)
print(f"ANÁLISE DE {selected_month_year}".center(60))
print("="*60)

# 🌟 Destaques rápidos
print("\n🔹 DESTAQUES DO MÊS")
if not df_month.empty:
    print(f"⚡ Atleta mais rápido: {max_speed_month['athlete']} – {max_speed_month['max_speed']:.1f} km/h")
else:
    print("⚡ Nenhuma pedalada registrada no mês.")

print(f"📏 Atleta que passou mais tempo pedalando: {top_athlete['athlete']} - {horas}h{minutos:02d}m")
print(f"📏 Distância total do grupo no mês: {distance_month:.0f} km")
print(f"🌍 Distância acumulada em {current_year}: {dist_total_year:.0f} km "
      f"({percentual:.2%} de uma volta ao mundo)")
# 🎉 Curiosidades do mês
print("\n🔹 CURIOSIDADES")
#print(f"🎉 {selected_month_year}")
for phrase in curiosity:
    print("•", phrase)

# 🏁 Top 3 pedaladas
print("\n🔹 TOP 3 PEDALADAS MAIS LONGAS")
for idx, row in enumerate(top3.itertuples(), start=1):
    print(f"{idx}. {row.athlete} – {row.distance:.1f} km")

# 🥇 Ranking mensal
print("\n🔹 RANKING MENSAL POR DISTÂNCIA")
print(tabulate(ranking_month, headers='keys', tablefmt='github', showindex=False))

# 🏆 Ranking anual
print("\n🔹 RANKING ANUAL POR DISTÂNCIA")
print(tabulate(ranking_year, headers='keys', tablefmt='github', showindex=False))

# 😴 Menor atividade
print("\n🔹 MENOR NÚMERO DE ATIVIDADES NO MÊS")
for name, row in least.iterrows():
    print(f"😴 {name} – {int(row['activities'])} pedaladas, {row['total_km']:.1f} km")

# 📉 Menor presença nos fins de semana
print("\n🔹 MENOR PARTICIPAÇÃO NOS PEDAIS EM GRUPO")
for name, row in least_weekend.iterrows():
    print(f"📉 {name} – {int(row['weekend_rides'])} pedaladas aos fins de semana")





                     ANÁLISE DE 2025-08                     

🔹 DESTAQUES DO MÊS
⚡ Atleta mais rápido: Vinicius Cainelli – 54.8 km/h
📏 Atleta que passou mais tempo pedalando: Diego Galdino - 17h18m
📏 Distância total do grupo no mês: 1668 km
🌍 Distância acumulada em 2025: 5334 km (13.31% de uma volta ao mundo)

🔹 CURIOSIDADES
• ⛰️ Isso daria 188.5 montes Everest empilhando garrafas de Grolsch.
• 🐜 Daria pra fazer uma fila com 166,818,000 formigas marchando sem parar.

🔹 TOP 3 PEDALADAS MAIS LONGAS
1. Diego Galdino – 151.7 km
2. Vinicius Cainelli – 150.9 km
3. Thamiris Ramos – 150.8 km

🔹 RANKING MENSAL POR DISTÂNCIA
|   🏅 | athlete              |   distance |
|------|----------------------|------------|
|    1 | Diego Galdino        |     353.19 |
|    2 | Rodrigo Bonilha      |     347.42 |
|    3 | Jair de Souza Junior |     329.14 |
|    4 | Thamiris Ramos       |     322.38 |
|    5 | Vinicius Cainelli    |     302.91 |
|    6 | Jonathan Machado     |      13.14 |

🔹 RANKING ANUAL 